# `cell_sim` on Colab — JCVI-Syn3A event-driven simulator

Multi-scale event-driven simulation of the JCVI-Syn3A minimal cell, built on real kinetic data from the Luthey-Schulten Lab.

This notebook walks through four progressively richer simulation modes:

1. **Real Syn3A baseline** — 458 proteins, enzymes turn over at real k_cat (no metabolite coupling)
2. **Priority 1.5** (reversibility + uptake) — reversible Michaelis-Menten, medium uptake, steady-state metabolism
3. **Priority 2** (gene expression) — add transcription, translation, mRNA degradation
4. **Priority 3** (novel substrates) — atomic-physics-backed k_cat for compounds not in the measured kinetic database

Modes 1–3 produce MP4s. Mode 4 produces text output showing the simulator handling a compound with no measured kinetic data.

**Runtime** → Change runtime type → High-RAM CPU is fine (GPU not used by the simulator itself; optional for MACE-OFF).

**Expected total time**: ~7–10 minutes on the updated simulator (previously 15–30 min).

## 1. Clone your fork and the upstream data

In [ ]:
# Clone the repo. Code lives in cell_sim/ inside Nikku03/cell.
import os, sys
REPO_OWNER = "Nikku03"
REPO_NAME  = "cell"         # change if you fork to a different name
SUBDIR     = "cell_sim"
REPO_ROOT  = f"/content/{REPO_NAME}"
WORK_DIR   = f"{REPO_ROOT}/{SUBDIR}"

# Escape the CWD before we potentially delete it (re-runs land inside WORK_DIR)
os.chdir('/content')

# Fresh clone every time — avoids stale caches when you push updates
if os.path.isdir(REPO_ROOT):
    !rm -rf {REPO_ROOT}
!git clone https://github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_ROOT}

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f'CWD: {os.getcwd()}')

# Clone the Luthey-Schulten upstream data (real Syn3A kinetics)
if not os.path.isdir('data/Minimal_Cell_ComplexFormation'):
    !mkdir -p data
    !cd data && git clone --depth 1 https://github.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation.git
!ls -la data/Minimal_Cell_ComplexFormation/input_data/

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!which ffmpeg || apt-get install -y ffmpeg

## 3. Verify data loading

In [ ]:
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
from pathlib import Path
from layer0_genome.syn3a_real import build_real_syn3a_cellspec
from layer3_reactions.sbml_parser import parse_sbml
from layer3_reactions.kinetics import load_all_kinetics, load_medium

spec, counts, complexes, kcats = build_real_syn3a_cellspec()
sbml = parse_sbml(Path('data/Minimal_Cell_ComplexFormation/input_data/Syn3A_updated.xml'))
kinetics = load_all_kinetics()
medium = load_medium()

print(f'\nSummary:')
print(f'  Genome: {spec.genome_size_bp:,} bp, {len(spec.proteins)} genes')
print(f'  Proteins with counts: {len(counts)} (total molecules: {sum(counts.values()):,})')
print(f'  Known complexes: {len(complexes)}')
print(f'  Metabolic SBML: {len(sbml.species)} species, {len(sbml.reactions)} reactions')
print(f'  Kinetic parameters: {len(kinetics)} reactions ({sum(1 for k in kinetics.values() if k.is_reversible)} reversible)')
print(f'  Medium: {len(medium)} external species')

## 4. Real Syn3A baseline (fastest, ~25 s wall including render)

In [ ]:
!python tests/render_real_syn3a.py

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('data/real_syn3a_movie/real_syn3a.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 5. Priority 1.5 — reversibility + medium uptake

Full central metabolism with reversible Michaelis-Menten kinetics. With the vectorized simulator, the 1.0 s biological simulation now takes ~4 s wall, down from ~49 s.

In [ ]:
!python tests/render_priority_15.py

In [ ]:
mp4 = open('data/priority_15_movie/priority_15.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 6. Priority 2 — full central dogma

Adds transcription, translation, mRNA degradation on top of Priority 1.5. 1.5 s biological time in ~7 s wall with the fast simulator.

In [ ]:
!python tests/render_priority_2.py

In [ ]:
mp4 = open('data/priority_2_movie/priority_2.mp4', 'rb').read()
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

## 7. Priority 3 — atomic k_cat for novel substrates

Priority 3 is the architectural move that sets this simulator apart: **given just a SMILES string and a target enzyme, the simulator computes a k_cat from first principles and fires real catalysis events**.

Two backends behind a common `AtomicEngine` interface:

- **`SimilarityBackend`** (always available, fast, well-calibrated): RDKit Morgan-2 fingerprints + Tanimoto similarity. Finds the most similar known substrate and scales its measured k_cat by similarity². Benchmarked on 143 Syn3A reactions: **50% predicted within 5×, 64% within 10×** of measured k_cat.
- **`MACEBackend`** (optional, needs `mace-torch`, ideally GPU): MACE-OFF pretrained ML force field → weakest O–H bond dissociation energy → Hammond + Eyring → k_cat. Experimental; currently uncalibrated against measured kinetics (produces rates that can be 2–3 orders of magnitude off). Stay on the similarity backend for demos.

The demo below adds **BrdU (5-bromo-2′-deoxyuridine)**, a real DNA proliferation tracer, to the cell and watches the simulator phosphorylate it via Syn3A's thymidine kinase (TMDK1), using a k_cat predicted entirely from BrdU's structural similarity to thymidine.

### 7.1 Install the Priority 3 dependencies

In [ ]:
# RDKit powers the similarity backend — required.
# ASE is needed for MACE-OFF; safe to install without enabling MACE.
!pip install -q rdkit ase

import rdkit, ase
print(f'RDKit {rdkit.__version__}')
print(f'ASE   {ase.__version__}')

# OPTIONAL — MACE-OFF atomic-physics backend.
# Heavy (~200 MB). Currently uncalibrated — predicts k_cats up to 1000x off
# for some substrates, producing runaway reactions in simulation. Leave
# commented unless you're actively benchmarking MACE vs similarity on a
# research basis.
# !pip install -q torch e3nn mace-torch

### 7.2 Run the BrdU demo

Two 300 ms simulations — baseline (no BrdU) and with 100,000 BrdU molecules added — printed side-by-side.

Expected: ~8 novel catalysis events via TMDK1, k_cat predicted at 10.36/s (Tanimoto 0.733 to thymidine).

In [ ]:
!python tests/demo_priority3.py

### 7.3 Try your own novel substrate

Replace the SMILES and target reaction below to see what the simulator does with a different compound. Some that work with the reference SMILES already loaded:

| Target enzyme | Real substrate | Try a novel analog |
|---|---|---|
| `TMDK1` (thymidine kinase) | thymidine | BrdU, 5-iododeoxyuridine (IdU), azidothymidine (AZT) |
| `GLYK` (glycerol kinase) | glycerol | propylene glycol, 1,3-propanediol, erythritol |
| `PFK` (phosphofructokinase) | fructose-6-phosphate | tagatose-6-phosphate, sorbose-6-phosphate |
| `DADNK` (deoxyadenosine kinase) | deoxyadenosine | 2,6-diaminopurine deoxyriboside |

In [ ]:
# Interactive novel-substrate experiment. Change these four variables.
NOVEL_NAME    = 'azidothymidine'
NOVEL_SMILES  = 'CC1=CN([C@H]2C[C@@H](N=[N+]=[N-])[C@H](CO)O2)C(=O)NC1=O'
TARGET_RXN    = 'TMDK1'
PRODUCT_NAME  = 'AZT_monophosphate'

import io, contextlib, time
import numpy as np
from pathlib import Path
from layer0_genome.syn3a_real import build_real_syn3a_cellspec
from layer2_field.dynamics import CellState
from layer2_field.fast_dynamics import FastEventSimulator
from layer2_field.real_syn3a_rules import (
    populate_real_syn3a, make_folding_rule, make_complex_formation_rules,
)
from layer3_reactions.sbml_parser import parse_sbml
from layer3_reactions.kinetics import load_all_kinetics, load_medium
from layer3_reactions.reversible import (
    build_reversible_catalysis_rules, initialize_medium,
)
from layer3_reactions.coupled import (
    initialize_metabolites, get_species_count, count_to_mM,
)
from layer1_atomic.engine import AtomicEngine
from layer3_reactions.novel_substrates import add_novel_substrate

with contextlib.redirect_stdout(io.StringIO()):
    spec, counts, complexes, _ = build_real_syn3a_cellspec()
sbml = parse_sbml(Path('data/Minimal_Cell_ComplexFormation/input_data/Syn3A_updated.xml'))
kinetics = load_all_kinetics()
medium = load_medium()
rev_rules, _ = build_reversible_catalysis_rules(sbml, kinetics)

state = CellState(spec)
populate_real_syn3a(state, counts, scale_factor=0.02, max_per_gene=10)
initialize_metabolites(state, sbml, cell_volume_um3=(4/3)*np.pi*0.2**3)
initialize_medium(state, medium)

engine = AtomicEngine(backend_name='similarity')
print(f'Atomic engine: {engine.active_backend}')

try:
    rule, info = add_novel_substrate(
        state, sbml, kinetics, engine,
        name=NOVEL_NAME, smiles=NOVEL_SMILES,
        target_reaction=TARGET_RXN, product_name=PRODUCT_NAME,
        initial_count=100_000, dead_end=True,
    )
except ValueError as e:
    print(f'Registration failed: {e}')
    rule = None

if rule is not None:
    print(f'\nRegistered {info.name}:')
    print(f'  nearest known:  {info.kcat_estimate.nearest_known_substrate}')
    if info.kcat_estimate.similarity is not None:
        print(f'  Tanimoto:       {info.kcat_estimate.similarity:.3f}')
    print(f'  predicted k_cat: {info.kcat_estimate.kcat_per_s:.3f} /s')
    print(f'  ({info.kcat_estimate.notes})')

    rules = ([make_folding_rule(20.0)] + rev_rules
             + make_complex_formation_rules(complexes, 0.05) + [rule])
    sim = FastEventSimulator(state, rules, mode='gillespie', seed=42)

    t0 = time.time()
    sim.run_until(t_end=0.3, max_events=1_000_000)
    print(f'\nSimulated 300 ms in {time.time()-t0:.1f} s wall')

    novel_consumed = 100_000 - get_species_count(state, info.atomic_species_id)
    product_made   = get_species_count(state, info.product_species_id)
    novel_events   = sum(1 for e in state.events if ':novel:' in e.rule_name)
    print(f'\nResult:')
    print(f'  {NOVEL_NAME:20s} consumed: {novel_consumed:,}')
    print(f'  {PRODUCT_NAME:20s} produced: {product_made:,}')
    print(f'  Novel catalysis events:   {novel_events}')
    if state.events and novel_events > 0:
        for e in state.events:
            if ':novel:' in e.rule_name:
                print(f'  Sample: t={e.time*1000:.2f}ms  {e.description}')
                break

## 8. Download the MP4s

In [ ]:
from google.colab import files
for path in [
    'data/real_syn3a_movie/real_syn3a.mp4',
    'data/priority_15_movie/priority_15.mp4',
    'data/priority_2_movie/priority_2.mp4',
]:
    if os.path.isfile(path):
        files.download(path)
    else:
        print(f'(skipping {path} - not generated yet)')

## 9. Interactive exploration

Run a short Priority 1.5 simulation directly and inspect metabolite changes. Uses the fast simulator — 300 ms in ~1.5 s wall.

In [ ]:
import io, contextlib, time
import numpy as np
from layer2_field.dynamics import CellState
from layer2_field.fast_dynamics import FastEventSimulator
from layer2_field.real_syn3a_rules import (
    populate_real_syn3a, make_folding_rule, make_complex_formation_rules,
)
from layer3_reactions.reversible import (
    build_reversible_catalysis_rules, initialize_medium,
)
from layer3_reactions.coupled import (
    initialize_metabolites, get_species_count, count_to_mM,
)

rev_rules, _ = build_reversible_catalysis_rules(sbml, kinetics)

state = CellState(spec)
populate_real_syn3a(state, counts, scale_factor=0.02, max_per_gene=10)
initialize_metabolites(state, sbml, cell_volume_um3=(4/3)*np.pi*0.2**3)
initialize_medium(state, medium)

rules = (
    [make_folding_rule(20.0)]
    + rev_rules
    + make_complex_formation_rules(complexes, base_rate_per_s=0.05)
)
sim = FastEventSimulator(state, rules, mode='gillespie', seed=42)
initial = dict(state.metabolite_counts)

print('Running 0.3s...')
t0 = time.time()
sim.run_until(t_end=0.3, max_events=500_000)
print(f'  {len(state.events):,} events in {time.time()-t0:.1f}s wall')

print('\nKey metabolite changes:')
for sid in ['M_atp_c', 'M_adp_c', 'M_g6p_c', 'M_pep_c', 'M_pyr_c',
            'M_lac__L_c', 'M_nadh_c', 'M_pi_c']:
    d = get_species_count(state, sid) - initial.get(sid, 0)
    dmM = count_to_mM(abs(d), state.metabolite_volume_L) * (1 if d >= 0 else -1)
    print(f'  {sid:14s}  delta={d:>+10,}  ({dmM:>+7.3f} mM)')

In [ ]:
# Forward vs reverse event counts for top-turnover reactions
from collections import Counter
import matplotlib.pyplot as plt

cat_counts = Counter()
for e in state.events:
    if e.rule_name.startswith('catalysis:'):
        parts = e.rule_name.split(':')
        rxn = parts[1]
        is_rev = len(parts) > 2 and parts[2] == 'rev'
        cat_counts[(rxn, is_rev)] += 1

rxn_totals = Counter()
for (rxn, _), n in cat_counts.items():
    rxn_totals[rxn] += n
top_rxns = [r for r, _ in rxn_totals.most_common(12)]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top_rxns))
fwd = [cat_counts.get((r, False), 0) for r in top_rxns]
rev = [cat_counts.get((r, True), 0) for r in top_rxns]
ax.bar(x - 0.2, fwd, 0.4, label='forward', color='#4dabf7')
ax.bar(x + 0.2, rev, 0.4, label='reverse', color='#f97316')
ax.set_xticks(x)
ax.set_xticklabels(top_rxns, rotation=45)
ax.set_ylabel('event count')
ax.set_title('Forward vs reverse flux for top-turnover reactions')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Priority 3 benchmark

Leave-one-out k_cat prediction on all 143 reactions in the Syn3A kinetic database, using the similarity backend. Produces per-reaction CSV + a log-log scatter plot.

In [ ]:
!python tests/benchmark_priority3.py

In [ ]:
from IPython.display import Image
Image('data/priority3_benchmark_scatter.png')

## 11. Whole-cell simulation at 100% scale

Everything so far has run at 2% scale (2,726 proteins, 100k–300k metabolites). That's honest for fast iteration but not a full cell. This section runs **100% of real Syn3A proteomics counts** — all 134,000+ proteins, full metabolite pools — for 1 second of biological time.

**Expected:**
- ~2 million catalysis events
- Wall time ≈ 20 minutes with the Rust backend, ≈ 30 minutes without
- Memory usage ≈ 2 GB (Colab High-RAM runtime required)

What this produces:
- `data/whole_cell/summary.txt` — final metabolite report
- `data/whole_cell/trajectory.csv` — key metabolites every 10 ms
- `data/whole_cell/events.csv` — per-rule event counts

If you need to cut runtime, use `WC_SCALE=0.5` for 50% scale (~7 min) or `WC_SCALE=0.25` for 25% (~2 min). Physics is unchanged at smaller scales — only stochastic noise is larger.

### 11.1 Install the Rust backend (optional, ~2x speedup)

If the Rust wheel from `phase3_rust.tar.gz` is in your uploads, this cell extracts and installs it. Skip this cell to run pure Python.

In [ ]:
# Install the Rust backend if the tarball is available.
# Upload phase3_rust.tar.gz via the Colab file browser first, OR clone from a release.
import os, sys, glob
rust_ok = False
tar_candidates = ['phase3_rust.tar.gz', '/content/phase3_rust.tar.gz',
                  '../phase3_rust.tar.gz']
found = None
for p in tar_candidates:
    if os.path.isfile(p):
        found = p
        break

if found:
    import subprocess
    print(f'Extracting {found}...')
    subprocess.check_call(['tar', '-xzf', found, '-C', '/tmp'])
    wheels = glob.glob('/tmp/cell_sim_rust/target/wheels/cell_sim_rust-*.whl')
    if wheels:
        print(f'Installing {wheels[0]}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               '--force-reinstall', wheels[0]])
        import cell_sim_rust
        print(f'Rust backend installed: v{cell_sim_rust.__version__}')
        rust_ok = True
    else:
        print('Tarball extracted but no wheel found. Running pure Python.')
else:
    print('phase3_rust.tar.gz not found. Running pure Python.')
    print('To use the Rust backend, upload phase3_rust.tar.gz to Colab first.')

### 11.2 Run the whole-cell simulation

This cell takes 20–30 minutes to complete at 100% scale. Colab streams progress every 10 wall-seconds so you can tell it's alive.

In [ ]:
import os

# Use Rust if we installed it above. Override scale / time via env vars if needed.
os.environ['WC_USE_RUST'] = '1' if rust_ok else '0'
os.environ['WC_SCALE']    = '1.0'       # 1.0 = 100%, try 0.25 or 0.5 for faster
os.environ['WC_T_END']    = '1.0'       # 1.0 seconds of biological time
os.environ['WC_SEED']     = '42'

# Run WITH the nutrient-uptake patch (GLCpts, O2t, FAt, GLYCt, lipid
# importers, synthetic nucleobase importers). Without these, the cell
# has no mouth for glucose etc. and decays over the 1s run.
os.environ['WC_WITH_UPTAKE'] = '1'

# Output goes to data/whole_cell/ — we rename after each run so we can
# keep both the fed-cell and the starving-cell trajectories side by side.
!rm -rf data/whole_cell
!python tests/render_whole_cell.py

# Preserve the fed-cell result before re-running without the patch.
!cp -r data/whole_cell data/whole_cell_fed
print('\n\n===== Saved fed-cell outputs to data/whole_cell_fed/ =====\n\n')


### 11.2b Comparison run: no-uptake (the starving cell)

Re-run the same 1-second simulation but with the uptake patch disabled, so you can see the decay side-by-side with the fed cell.


In [ ]:
# --- Optional: run the SAME scale again WITHOUT the uptake patch ---
# This reproduces the "starving cell" for the side-by-side comparison.
# Comment out the block below to skip (saves ~10-20 min wall time).

RUN_COMPARISON = True   # set False to skip the starving-cell run

if RUN_COMPARISON:
    os.environ['WC_WITH_UPTAKE'] = '0'
    !rm -rf data/whole_cell
    !python tests/render_whole_cell.py
    !cp -r data/whole_cell data/whole_cell_starving
    print('\n===== Saved starving-cell outputs to data/whole_cell_starving/ =====')
else:
    print('Skipping starving-cell run (RUN_COMPARISON = False).')


### 11.3 Inspect the trajectory

Plots the ATP/ADP ratio over the 1-second run. At full scale, metabolism reaches quasi-steady-state very quickly (within ~100 ms) and the ATP/ADP ratio stabilizes — exactly the behavior you'd expect from a healthy cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def load_traj(path):
    if os.path.isfile(path):
        return pd.read_csv(path)
    return None

fed = load_traj('data/whole_cell_fed/trajectory.csv')
starv = load_traj('data/whole_cell_starving/trajectory.csv')

if fed is None:
    # fall back to the live directory if the "_fed" copy doesn't exist
    fed = load_traj('data/whole_cell/trajectory.csv')

print(f'Fed cell trajectory:      {len(fed) if fed is not None else "(missing)"} points')
print(f'Starving cell trajectory: {len(starv) if starv is not None else "(missing)"} points')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

def plot_pair(ax, col, title, ylog=False):
    if fed is not None:
        ax.plot(fed.t_ms, fed[col], label='fed (uptake ON)', color='#2a9d8f', lw=2)
    if starv is not None:
        ax.plot(starv.t_ms, starv[col], label='starving (uptake OFF)',
                color='#e63946', lw=2, ls='--')
    ax.set_xlabel('biological time (ms)')
    ax.set_title(title)
    ax.legend(loc='best', fontsize=9)
    if ylog:
        ax.set_yscale('log')

plot_pair(axes[0,0], 'M_atp_c', 'ATP')
plot_pair(axes[0,1], 'M_amp_c', 'AMP (watch for runaway accumulation)')
plot_pair(axes[1,0], 'M_g6p_c', 'G6P (glycolysis entry point)')
plot_pair(axes[1,1], 'M_ppi_c', 'PPi (byproduct accumulation)')

plt.suptitle('Fed vs starving whole-Syn3A cell — 100% scale, 1 s biological time',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.savefig('data/whole_cell_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# Also show the ATP/ADP ratio
fig, ax = plt.subplots(figsize=(10, 5))
if fed is not None:
    ratio_fed = fed.M_atp_c / fed.M_adp_c.replace(0, 1)
    ax.plot(fed.t_ms, ratio_fed, label='fed (uptake ON)', color='#2a9d8f', lw=2)
if starv is not None:
    ratio_starv = starv.M_atp_c / starv.M_adp_c.replace(0, 1)
    ax.plot(starv.t_ms, ratio_starv, label='starving (uptake OFF)',
            color='#e63946', lw=2, ls='--')
ax.set_xlabel('biological time (ms)')
ax.set_ylabel('ATP / ADP')
ax.set_title('Adenylate energy charge proxy — a healthy cell holds this ~10-20')
ax.axhspan(8, 20, alpha=0.1, color='green', label='physiological range (8-20)')
ax.legend()
plt.tight_layout()
plt.savefig('data/whole_cell_atp_adp.png', dpi=100, bbox_inches='tight')
plt.show()


### 11.4 Event breakdown

Which reactions fired most? At full scale this is the real metabolic portrait of Syn3A — the 143-reaction kinetic database in action.

In [ ]:
# Event breakdown for the FED cell (with uptake). If you also ran the
# starving comparison, its events.csv is in data/whole_cell_starving/.
import os

ev_path = 'data/whole_cell_fed/events.csv' if os.path.isfile('data/whole_cell_fed/events.csv') \
          else 'data/whole_cell/events.csv'

ev = pd.read_csv(ev_path)
total = ev['count'].sum()
ev['pct'] = 100 * ev['count'] / total
print(f'Total events: {total:,}   ({ev_path})')
print(f'Unique reactions firing: {len(ev)}')
print()
print('Top 25:')
print(ev.head(25).to_string(index=False))

# Top-20 chart
top20 = ev.head(20).iloc[::-1]  # reverse for horizontal bar
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top20['rule'], top20['count'], color='#4dabf7')
# Highlight uptake rules in a different color
uptake_names = {'GLCpts', 'O2t', 'GLYCt', 'FAt', 'CHOLt', 'TAGt',
                'ADEt_syn', 'GUAt_syn', 'URAt_syn', 'CYTDt_syn'}
for b, name in zip(bars, top20['rule']):
    if name in uptake_names:
        b.set_color('#2a9d8f')
ax.set_xlabel('event count over 1 s biological time')
ax.set_title(f'Top-20 reactions by turnover — whole-cell Syn3A ({total:,} total events)\n'
             f'(green = nutrient-uptake rules from the patch)')
plt.tight_layout()
plt.savefig('data/whole_cell_top20.png', dpi=100, bbox_inches='tight')
plt.show()


### 11.5 Download whole-cell outputs

In [ ]:
from google.colab import files
for p in ['data/whole_cell_fed/summary.txt',
          'data/whole_cell_fed/trajectory.csv',
          'data/whole_cell_fed/events.csv',
          'data/whole_cell_starving/summary.txt',
          'data/whole_cell_starving/trajectory.csv',
          'data/whole_cell_starving/events.csv',
          'data/whole_cell_comparison.png',
          'data/whole_cell_atp_adp.png',
          'data/whole_cell_top20.png']:
    if os.path.isfile(p):
        files.download(p)
        print(f'downloaded: {p}')
    else:
        print(f'(skip) not found: {p}')
